# LLM pretraining data — interactive workbench

Notebook interface to the mixture-data pipeline (replaces the `browse.py` /
`tokenize_source.py` / `train_tokenizer.py` / `build_mixture.py` CLIs). Each
section mirrors one script; edit the config variables and run the cell instead
of passing flags. The heavy lifting still lives in the sibling modules
(`sources.py`, `tokenize_source.py`, `build_mixture.py`, `train_tokenizer.py`) —
this notebook just drives them.

**Sections**
1. Setup
2. The mixture — list & category weights
3. Training plan — target tokens & repeat factors
4. Mixture flow — Sankey diagram
5. Peek at real samples
6. Document-length distributions
7. Tokenize sources into caches  (`tokenize_source.py`)
8. Train a custom BPE tokenizer  (`train_tokenizer.py`)
9. Build the packed mixture  (`build_mixture.py`)

## 1. Setup

In [ ]:
import os
import sys
from pathlib import Path

# Datasets/models/caches live on the big volume; set before importing hf libs.
os.environ["HF_HOME"] = "/mnt/ai/data/hf"

# The pipeline modules live next to this notebook; import them directly.
sys.path.insert(0, str(Path.cwd()))

import pandas as pd

from sources import SOURCES, TARGET_TOKENS, category_weights, get, plan

print(
    f"{len(SOURCES)} sources registered; default budget {TARGET_TOKENS / 1e9:.0f}B tokens"
)

## 2. The mixture — list & category weights

The mixture registry: every registered slice with its pretrain
weight and category, then the per-category rollup.

In [ ]:
rows = [
    {
        "key": s.key,
        "source": s.title,
        "category": s.category,
        "weight": s.weight,
        "sft_weight": s.sft_weight,
        "avail_tokens": s.avail_tokens,
        "deferred": s.deferred,
    }
    for s in SOURCES
]
mix = pd.DataFrame(rows)
display(
    mix.style.format(
        {"weight": "{:.0%}", "sft_weight": "{:.0%}", "avail_tokens": "{:,.0f}"}
    ).set_caption("Planned pretraining mixture")
)

cats = category_weights()
cat_df = (
    pd.Series(cats, name="weight")
    .sort_values(ascending=False)
    .to_frame()
    .assign(weight=lambda d: d["weight"])
)
cat_df.loc["TOTAL"] = cat_df["weight"].sum()
print("By category:")
display(cat_df.style.format({"weight": "{:.0%}"}))

## 3. Training plan — target tokens & repeat factors

Target tokens & repeat factors for a token budget. For each source: target tokens
(`weight × budget`) and the **repeat factor** (`target ÷ unique available`).
A repeat > 1 means the slice must be seen more than once; > ~3 in pretraining is
a red flag. Set `BUDGET` to any token count.

In [ ]:
BUDGET = TARGET_TOKENS  # e.g. 1e9, 10e9, 20e9

plan_df = pd.DataFrame(plan(int(BUDGET)))
plan_df["repeat"] = plan_df["target_tokens"] / plan_df["avail_tokens"]
display(
    plan_df[["key", "category", "weight", "target_tokens", "avail_tokens", "repeat"]]
    .style.format(
        {
            "weight": "{:.0%}",
            "target_tokens": "{:,.0f}",
            "avail_tokens": "{:,.0f}",
            "repeat": "{:.2f}x",
        }
    )
    .set_caption(f"Training plan for {BUDGET / 1e9:.0f}B tokens")
)

roll = (
    plan_df.groupby("category")[["target_tokens", "avail_tokens"]]
    .sum()
    .assign(repeat=lambda d: d["target_tokens"] / d["avail_tokens"])
    .sort_values("target_tokens", ascending=False)
)
print("By category (repeat > 3x = heavy repetition):")
display(
    roll.style.format(
        {"target_tokens": "{:,.0f}", "avail_tokens": "{:,.0f}", "repeat": "{:.1f}x"}
    )
)

## 4. Mixture flow — Sankey diagram

A visual of the pretrain mixture: total budget → category → source, with band
widths proportional to weight. Only pretrain-weighted sources are shown
(`ultrachat` is SFT-only, `weight=0`).

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.path import Path as MPath

PAL = {
    "code": "#4C78A8",
    "web": "#59A14F",
    "math": "#E45756",
    "tools": "#B279A2",
    "chat": "#9D7660",
}
GAP, BW = 0.012, 0.03  # node gap (weight units) / bar width (x units)
X = {"total": 0.0, "cat": 1.0, "src": 2.0}

srcs = [s for s in SOURCES if s.weight > 0]  # pretrain mix only
cats = {}
for s in srcs:
    cats.setdefault(s.category, []).append(s)
cat_order = sorted(cats, key=lambda c: -sum(s.weight for s in cats[c]))
cat_w = {c: sum(s.weight for s in cats[c]) for c in cat_order}
H = sum(cat_w.values())
n_src = len(srcs)


def ribbon(ax, x0, x1, y0t, y0b, y1t, y1b, color, alpha=0.45):
    mx = (x0 + x1) / 2
    verts = [
        (x0, y0t),
        (mx, y0t),
        (mx, y1t),
        (x1, y1t),
        (x1, y1b),
        (mx, y1b),
        (mx, y0b),
        (x0, y0b),
        (x0, y0t),
    ]
    codes = [
        MPath.MOVETO,
        MPath.CURVE4,
        MPath.CURVE4,
        MPath.CURVE4,
        MPath.LINETO,
        MPath.CURVE4,
        MPath.CURVE4,
        MPath.CURVE4,
        MPath.CLOSEPOLY,
    ]
    ax.add_patch(
        patches.PathPatch(
            MPath(verts, codes), facecolor=color, edgecolor="none", alpha=alpha
        )
    )


# node spans (top, bottom), stacked top-down
cat_span, src_span = {}, {}
y = H + GAP * (len(cat_order) - 1)
for c in cat_order:
    cat_span[c] = (y, y - cat_w[c])
    y -= cat_w[c] + GAP
y = H + GAP * (n_src - 1)
for c in cat_order:
    for s in cats[c]:
        src_span[s.key] = (y, y - s.weight)
        y -= s.weight + GAP

fig, ax = plt.subplots(figsize=(11, 7))
tot_top = H
for c in cat_order:
    col = PAL[c]
    ct, cb = cat_span[c]
    ribbon(ax, X["total"] + BW, X["cat"], tot_top, tot_top - cat_w[c], ct, cb, col)
    tot_top -= cat_w[c]
    ctop = ct
    for s in cats[c]:
        st, sb = src_span[s.key]
        ribbon(ax, X["cat"] + BW, X["src"], ctop, ctop - s.weight, st, sb, col)
        ctop -= s.weight

ax.add_patch(patches.Rectangle((X["total"], 0), BW, H, color="#555"))
ax.text(
    X["total"] - 0.03,
    H / 2,
    f"mixture\n{H:.0%}",
    ha="right",
    va="center",
    fontsize=10,
    weight="bold",
)
for c in cat_order:
    t, b = cat_span[c]
    ax.add_patch(patches.Rectangle((X["cat"], b), BW, t - b, color=PAL[c]))
    ax.text(
        X["cat"] + BW + 0.03,
        (t + b) / 2,
        f"{c}  {cat_w[c]:.0%}",
        ha="left",
        va="center",
        fontsize=9,
        weight="bold",
    )
for s in srcs:
    t, b = src_span[s.key]
    ax.add_patch(patches.Rectangle((X["src"], b), BW, t - b, color=PAL[s.category]))
    ax.text(
        X["src"] + BW + 0.03,
        (t + b) / 2,
        f"{s.key}  {s.weight:.0%}",
        ha="left",
        va="center",
        fontsize=8,
    )

ax.set_xlim(-0.5, X["src"] + 0.9)
ax.set_ylim(-0.02, H + GAP * (n_src - 1) + 0.02)
ax.axis("off")
ax.set_title(
    "Pretraining mixture: budget → category → source", fontsize=12, weight="bold"
)
plt.tight_layout()
plt.show()

## 5. Peek at real samples

Streams a handful of
real rows straight from the backing dataset — downloading only the smallest
shard (cached under `HF_HOME`), or fetching Stack v2 content on demand from
Software Heritage S3. Set `SOURCE = None` to sweep every non-deferred slice.

In [ ]:
SOURCE = "finemath"  # a source key from section 2, or None to preview all
N = 3  # rows per source
CHARS = 1500  # truncate each sample to this many chars
FULL = False  # True = print samples untruncated


def show_source(key, n=N, chars=CHARS, full=FULL):
    src = get(key)
    print(f"\n=== {src.key}  |  {src.title}  ({src.weight:.0%}, {src.category}) ===")
    print(f"repo={src.hf_repo or '—'}  config={src.config}  split={src.split}")
    print(src.notes)
    if src.deferred:
        print("  deferred — no data to show yet.")
        return
    print("-" * 80)
    try:
        for i, sample in enumerate(src.sample(n)):
            meta = " ".join(f"{k}={v}" for k, v in sample.meta.items() if v is not None)
            print(f"\n[{key} #{i}] {meta}")
            text = sample.text
            if not full and len(text) > chars:
                text = text[:chars] + f"\n... [+{len(sample.text) - chars} chars]"
            print(text)
        sh = src.last_shard
        if sh is not None:
            state = "cached" if sh.cached else "downloaded once, now cached"
            print(
                f"\n(read from shard {sh.path.split('/')[-1]}, {sh.size / 1e6:.0f} MB — {state}; not the full dataset)"
            )
    except Exception as e:
        print(f"  error reading {key}: {e}")


if SOURCE is None:
    for s in SOURCES:
        if not s.deferred:
            show_source(s.key, n=1, chars=800)
else:
    show_source(SOURCE)

## 6. Document-length distributions (tokens)

Reads the already-tokenized `tok/<key>/ids.bin` caches from section 7 (docs are
`<|endoftext|>`-separated) and plots per-document **token** length — the unit
that actually matters for packing/context. Shows a per-source stats table
(mean, median/p50, p90, p99) and a boxplot.

Reads only the first `MAX_TOKENS_PER_SOURCE` tokens of each cache (plenty of docs
for a stable distribution), so it's fast and touches only local disk.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from build_mixture import _tok_dir
from chimera.tokenizers import BPETokenizer

TOK_ROOT = _tok_dir(sft=False)  # /mnt/ai/data/llm-mix/tok
MAX_TOKENS_PER_SOURCE = 100_000_000  # cap read per source (None = whole cache)
INCLUDE = None  # list of source keys, or None for all non-deferred
TOKENIZER = (
    "LiquidAI/LFM2.5-230M"  # must match the tokenizer the caches were built with
)

CAT_COLORS = {
    "code": "#4C78A8",
    "web": "#59A14F",
    "math": "#E45756",
    "tools": "#B279A2",
    "chat": "#9D7660",
}

eos = BPETokenizer.from_pretrained(TOKENIZER)._tok.token_to_id("<|endoftext|>")
targets = (
    [get(k) for k in INCLUDE] if INCLUDE else [s for s in SOURCES if not s.deferred]
)

lengths, stat_rows = {}, []
for s in targets:
    ids_path = TOK_ROOT / s.key / "ids.bin"
    if not ids_path.exists():
        print(f"  skip {s.key}: no cache (run section 7)")
        continue
    ids = np.memmap(ids_path, dtype=np.uint16, mode="r")
    n = (
        len(ids)
        if MAX_TOKENS_PER_SOURCE is None
        else min(len(ids), MAX_TOKENS_PER_SOURCE)
    )
    chunk = np.asarray(ids[:n])
    ends = np.flatnonzero(chunk == eos)  # doc boundaries (trailing eos per doc)
    if len(ends) < 2:
        print(f"  skip {s.key}: too few docs in the read window")
        continue
    L = np.diff(np.concatenate(([-1], ends))) - 1  # tokens per doc, excluding the eos
    lengths[s.key] = L
    stat_rows.append(
        {
            "key": s.key,
            "category": s.category,
            "n_docs": len(L),
            "mean": L.mean(),
            "median": np.median(L),
            "p50": np.percentile(L, 50),
            "p90": np.percentile(L, 90),
            "p99": np.percentile(L, 99),
            "min": int(L.min()),
            "max": int(L.max()),
        }
    )
    print(f"  {s.key}: {len(L):,} docs from {n / 1e6:.0f}M tokens", flush=True)

stats = pd.DataFrame(stat_rows).set_index("key")
display(
    stats.style.format(
        {
            c: "{:,.0f}"
            for c in ["n_docs", "mean", "median", "p50", "p90", "p99", "min", "max"]
        }
    ).set_caption("Document length (tokens) per source")
)

# horizontal boxplot, log scale, colored by category
keys = list(lengths)
fig, ax = plt.subplots(figsize=(11, 0.5 * len(keys) + 2))
bp = ax.boxplot(
    [lengths[k] for k in keys],
    vert=False,
    patch_artist=True,
    tick_labels=keys,
    showfliers=False,
    medianprops=dict(color="black"),
)
for patch, k in zip(bp["boxes"], keys):
    patch.set_facecolor(CAT_COLORS.get(get(k).category, "#888"))
    patch.set_alpha(0.7)
ax.set_xscale("log")
ax.set_xlabel("document length (tokens, log scale)")
ax.set_title(
    "Document-length distribution per source (tokens)", fontsize=12, weight="bold"
)
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

# per-source histograms (small multiples), log-spaced bins
ncol = 3
nrow = -(-len(keys) // ncol)
fig, axes = plt.subplots(nrow, ncol, figsize=(4 * ncol, 2.4 * nrow), squeeze=False)
for ax, k in zip(axes.ravel(), keys):
    L = lengths[k]
    bins = np.logspace(np.log10(max(L.min(), 1)), np.log10(L.max()), 40)
    ax.hist(L, bins=bins, color=CAT_COLORS.get(get(k).category, "#888"), alpha=0.8)
    ax.axvline(np.median(L), color="black", lw=1, ls="--")
    ax.set_xscale("log")
    ax.set_title(f"{k}  (median {np.median(L):,.0f})", fontsize=9)
    ax.tick_params(labelsize=7)
for ax in axes.ravel()[len(keys) :]:
    ax.axis("off")
fig.supxlabel("document length (tokens, log scale)", fontsize=10)
fig.suptitle(
    "Token-length histograms per source (dashed = median)", fontsize=12, weight="bold"
)
plt.tight_layout()
plt.show()

## 7. Tokenize sources into caches

Equivalent to `tokenize_source.py`. Renders each row per its `kind`, tokenizes
with the chosen BPE, appends `<|endoftext|>` between documents, and writes a flat
`uint16` stream to `/mnt/ai/data/llm-mix/tok/<key>/` (or `tok_sft/` with masks
when `SFT=True`). Per-source caps come from `weight × BUDGET`; a complete cache is
skipped on re-run.

> ⚠️ This downloads/streams real data and can take a long time (Stack v2 pulls
> content from S3). Start with a small `MAX_TOKENS` smoke test. Set
> `SELECTED = None` to run every non-deferred source.

In [ ]:
from tokenize_source import tokenize_source, tokenize_source_sft
import math

SELECTED = ["finemath"]  # list of source keys, or None for all non-deferred
BUDGET = 10e9  # total mixture budget; per-source cap = weight * budget
MAX_TOKENS = (
    5_000_000  # hard per-source cap override for smoke tests; None = use budget
)
SFT = False  # True = masked ChatML tokenization (chat/openmath only)
TOKENIZER = "LiquidAI/LFM2.5-230M"  # HF id or local path to a custom tokenizer
S3_WORKERS = 128  # parallel SWH S3 fetchers (stackv2)


def cap_for(src):
    if MAX_TOKENS is not None:
        return int(MAX_TOKENS)
    w = src.sft_weight if (SFT and src.sft_weight is not None) else src.weight
    return int(math.ceil(w * BUDGET))


if SELECTED is None:
    targets = [s for s in SOURCES if not s.deferred]
    if SFT:
        targets = [s for s in targets if s.kind in ("chat", "openmath")]
else:
    targets = [get(k) for k in SELECTED]

for src in targets:
    if SFT:
        tokenize_source_sft(src.key, cap_for(src), tokenizer=TOKENIZER)
    else:
        tokenize_source(
            src.key, cap_for(src), s3_workers=S3_WORKERS, tokenizer=TOKENIZER
        )

## 8. Train a custom BPE tokenizer

Equivalent to `train_tokenizer.py`. Samples a *weighted* text corpus straight
from the mixture (same per-`kind` rendering as tokenization, Stack v2 content
included), then trains a byte-level BPE with the ChatML special tokens baked in,
writing to `/mnt/ai/data/llm-mix/tokenizer/<name>/`. The corpus is sampled once
and cached, so a whole suite of vocab sizes trains from the identical sample
without re-hitting S3/HF.

> ⚠️ Sampling the corpus is the expensive part. Use a small `TOTAL_CHARS` +
> `NO_CODE=True` for a quick smoke; size `TOTAL_CHARS` to your largest vocab.

In [ ]:
import json
import train_tokenizer as TT

NAME = "llm-bpe"  # base name; suite -> <name>-<tag> (e.g. llm-bpe-4k)
VOCAB_SIZES = [8192]  # one or more sizes trained from the SAME corpus (each <= 65535)
TOTAL_CHARS = 20e6  # total corpus chars, split by mixture weight
MIN_FREQUENCY = 2  # min pair count for a merge
SOURCE_KEYS = None  # restrict to these keys, or None for all non-deferred
NO_CODE = True  # skip the slow S3-backed Stack v2 sources
S3_WORKERS = 64
RESAMPLE = False  # re-sample even if a cached corpus exists
DROP_CORPUS = False  # delete the cached corpus after training
EVAL = False  # report compression vs BASELINE after training
BASELINE = "LiquidAI/LFM2.5-230M"  # incumbent tokenizer for --eval comparison

for v in VOCAB_SIZES:
    assert v <= TT.UINT16_MAX, f"vocab {v} exceeds uint16 ceiling {TT.UINT16_MAX}"
sizes = sorted(set(VOCAB_SIZES))

targets = (
    [get(k) for k in SOURCE_KEYS]
    if SOURCE_KEYS
    else [s for s in SOURCES if not s.deferred]
)
if NO_CODE:
    targets = [s for s in targets if s.kind != "stackv2"]
assert targets, "no sources selected"

TT.OUT_ROOT.mkdir(parents=True, exist_ok=True)
corpus_path = TT.OUT_ROOT / f"{NAME}-corpus.jsonl"
meta_path = TT.OUT_ROOT / f"{NAME}-corpus.meta.json"
print(
    f"suite '{NAME}': vocabs {sizes} from ~{TOTAL_CHARS / 1e9:.2f}B chars over "
    f"{len(targets)} sources ({', '.join(s.key for s in targets)})"
)
print(f"specials: {TT.DEFAULT_SPECIALS}\n")

# 1) sample the corpus once (or reuse the cache)
if corpus_path.exists() and not RESAMPLE:
    stat = json.loads(meta_path.read_text())
    print(
        f"reusing cached corpus {corpus_path} ({stat['n_chars'] / 1e9:.2f}B chars, {stat['n_docs']:,} docs)\n"
    )
else:
    corpus = TT.Corpus(targets, int(TOTAL_CHARS), S3_WORKERS)
    stat = TT._materialize_corpus(corpus, corpus_path)
    meta_path.write_text(json.dumps(stat, indent=2))
    print(
        f"\ncorpus cached: {stat['n_chars'] / 1e9:.2f}B chars, {stat['n_docs']:,} docs -> {corpus_path}\n"
    )

provenance = {
    "corpus_name": NAME,
    "total_chars_requested": int(TOTAL_CHARS),
    "realized_corpus_chars": stat["n_chars"],
    "realized_chars_by_source": stat.get("realized_chars", {}),
    "sources": [s.key for s in targets],
}

# 2) train every vocab size from the cached corpus
for v in sizes:
    name = NAME if len(sizes) == 1 else f"{NAME}-{TT._vocab_tag(v)}"
    TT._train_one(
        v,
        lambda: TT._read_corpus(corpus_path),
        MIN_FREQUENCY,
        TT.OUT_ROOT / name,
        provenance,
    )
    if EVAL:
        TT._eval_compression(
            TT.OUT_ROOT / name / "tokenizer.json",
            targets,
            BASELINE,
            sample_chars=5_000_000,
        )

if DROP_CORPUS:
    corpus_path.unlink(missing_ok=True)
    print(f"dropped cached corpus {corpus_path}")
else:
    print(
        f"\ncached corpus kept at {corpus_path} (re-run with more VOCAB_SIZES to reuse it)"
    )

## 9. Build the packed mixture

Equivalent to `build_mixture.py`. Reads the per-source `tok/<key>/` caches from
section 5, takes each source's share of `TOTAL` tokens (weights renormalized over
whichever sources have a *complete* cache), block-shuffles the pieces so no source
sits in one contiguous run, and writes `train.bin` / `val.bin` / `manifest.json`
to `/mnt/ai/data/llm-mix/mix/<name>/` (or `mix_sft/` with masks when `SFT=True`).

In [ ]:
from build_mixture import build, available_sources

NAME = "mix_1B"  # output mix name
TOTAL = 1e9  # total tokens to pack
VAL_FRAC = 0.005  # fraction held out (contiguous tail per source)
SEED = 42
SFT = False  # True = pack SFT caches (ids + supervise mask) from tok_sft/

avail = available_sources(sft=SFT)
if not avail:
    print("no tokenized sources found — run section 5 first")
else:
    print("available caches:", ", ".join(f"{k}({n / 1e6:.0f}M)" for k, _, n in avail))
    build(NAME, int(TOTAL), VAL_FRAC, SEED, sft=SFT)